In [1]:
import sys
import os

zero123_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'zero123')
color_controlnet_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'color_controlnet')
controlnet_dir = os.path.join("/home/vaibhav", 'diffusion_augmentation', 'controlnet')
device = "cuda"
color_control_device = device
zero123_device = device
control_net_device = device

sys.path.append(color_controlnet_dir)
#sys.path.append(zero123_dir)
sys.path.append(controlnet_dir)

from color_controlnet.diffusers import ControlNetModel, LineartDetector, StableDiffusionImg2ImgControlNetPalettePipeline
from color_controlnet.diffusers import UniPCMultistepScheduler
from color_controlnet.infer_palette import get_cond_color, show_anns, image_grid, HWC3, resize_in_buckets, SAMImageAnnotator
from color_controlnet.infer_palette_img2img import control_color_augment

# # Zero123 imports
# from zero123.nerf import load_model_from_config, generate_angles
# from zero123.ldm.util import create_carvekit_interface
# from zero123.ldm.models.diffusion.ddim import DDIMSampler as Zero123DDIMSampler
# from omegaconf import OmegaConf

# ControlNet imports
import torch
import numpy as np
import cv2
from PIL import Image
import random
import einops
from torchvision.transforms.functional import to_pil_image
from transformers import pipeline
from controlnet.annotator.util import resize_image, HWC3
from controlnet.annotator.canny import CannyDetector
from controlnet.annotator.uniformer import UniformerDetector
from controlnet.annotator.midas import MidasDetector
from torchvision.transforms.functional import to_pil_image

from controlnet.cldm.model import create_model, load_state_dict
from controlnet.cldm.ddim_hacked import DDIMSampler as ControlNetDDIMSampler
from pytorch_lightning import seed_everything

#TODO: figure out how to add all the sys paths without control net imports failing when zero123 imports are not commented out


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/pytorch_lightning/plugins/training_type/ddp.py:68: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import DistributedOptimizer


In [2]:
def initialize_color_controlnet_models():
    """
    Initialize all models and detectors used in the pipeline
    Returns:
        control_net: dict, containing all ControlNet models and detectors
        llava: dict, containing the LLAVA model and processor
        zero123: dict, containing the Zero123 model and Carvekit interf`ace
        color_control: dict, containing the Color Control model and SAM annotator

    """

    # Color Control model
    print('Loading Color Control model...')
    color_control = {}

    controlnet = ControlNetModel.from_config("./model_configs/controlnet_config.json").half()
    adapter = ControlNetModel.from_config("./model_configs/controlnet_config.json").half()

    sketch_method = "skmodel"
    sam_annotator = SAMImageAnnotator()

    model_ckpt = f"./models/color_img2img_palette.pt"
    model_sd = torch.load(model_ckpt, map_location="cpu")["module"]

    # assign the weights of the controlnet and adapter separately
    controlnet_sd = {}
    adapter_sd = {}
    for k in model_sd.keys():
        if k.startswith("controlnet"):
            controlnet_sd[k.replace("controlnet.", "")] = model_sd[k]
        if k.startswith("adapter"):
            adapter_sd[k.replace("adapter.", "")] = model_sd[k]

    msg_control = controlnet.load_state_dict(controlnet_sd, strict=True)
    print(f"msg_control: {msg_control} ")
    if adapter is not None:
        msg_adapter = adapter.load_state_dict(adapter_sd, strict=False)
        print(f"msg_adapter: {msg_adapter} ")

    # define the inference pipline
    # sdv15_path = "/home/pat/diffusion_augmentation/color_controlnet/model_configs/sd15_config.json"
    pipe = StableDiffusionImg2ImgControlNetPalettePipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        adapter=adapter,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(color_control_device)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    color_control['pipe'] = pipe
    color_control['sam_annotator'] = sam_annotator
    color_control['adapter'] = adapter 

    return color_control

In [3]:
color_control_model = initialize_color_controlnet_models()
color_control_model

Loading Color Control model...


/home/vaibhav/diffusion_augmentation/color_controlnet/diffusers/configuration_utils.py:195: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'color_controlnet.diffusers.models.controlnet.ControlNetModel'>.load_config(...) followed by <class 'color_controlnet.diffusers.models.controlnet.ControlNetModel'>.from_config(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)
/home/vaibhav/diffusion_augmentation/color_controlnet/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/

msg_control: <All keys matched successfully> 
msg_adapter: <All keys matched successfully> 


Fetching 15 files: 100%|██████████| 15/15 [00:22<00:00,  1.50s/it]
You have disabled the safety checker for <class 'color_controlnet.diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img_controlnet_palette.StableDiffusionImg2ImgControlNetPalettePipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
The config attributes {'skip_prk_steps': True, 'set_alpha_to_one': False, 'steps_offset': 1, 'clip_sample': False} were passed to UniPCMultistepScheduler, but are not expected and will be ignored. Please verify yo

{'pipe': StableDiffusionImg2ImgControlNetPalettePipeline {
   "_class_name": "StableDiffusionImg2ImgControlNetPalettePipeline",
   "_diffusers_version": "0.15.0.dev0",
   "adapter": [
     "models",
     "ControlNetModel"
   ],
   "controlnet": [
     "models",
     "ControlNetModel"
   ],
   "feature_extractor": [
     "transformers",
     "CLIPImageProcessor"
   ],
   "requires_safety_checker": true,
   "safety_checker": [
     null,
     null
   ],
   "scheduler": [
     "diffusers",
     "PNDMScheduler"
   ],
   "text_encoder": [
     "transformers",
     "CLIPTextModel"
   ],
   "tokenizer": [
     "transformers",
     "CLIPTokenizer"
   ],
   "unet": [
     "diffusers",
     "UNet2DConditionModel"
   ],
   "vae": [
     "diffusers",
     "AutoencoderKL"
   ]
 },
 'sam_annotator': <color_controlnet.infer_palette.SAMImageAnnotator at 0x7d081a31b850>,
 'adapter': ControlNetModel(
   (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (time_proj): Timestep

In [4]:
from IPython.display import display
import cv2

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

for class_name in classes:
    input_directory = os.path.join("torch/caltech256/256_ObjectCategories", class_name)
    output_directory = os.path.join("augmented_images/caltech256/color_controlnet", class_name)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    count = 0
    for file in os.listdir(input_directory):
        if count >= 2:
            break
        
        img = Image.open(os.path.join(input_directory, file))

        caption = class_name.split(".")[-1]
        color_augmented = control_color_augment(img, color_control_model['adapter'], color_control_model['pipe'], caption, color_control_model['sam_annotator'], 1, color_control_device)

        augmented_image = color_augmented[0]
        output_path = os.path.join(output_directory, file)

        augmented_image.save(output_path)
        print("Writing " + output_path + "...")

        count += 1

    

/home/vaibhav/diffusion_augmentation/color_controlnet/diffusers/models/SKmodel.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(modelpa

Writing augmented_images/caltech256/color_controlnet/001.ak47/001_0072.jpg...


100%|██████████| 22/22 [00:01<00:00, 16.13it/s]


Writing augmented_images/caltech256/color_controlnet/001.ak47/001_0003.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.19it/s]


...

Writing augmented_images/caltech256/color_controlnet/004.baseball-bat/004_0049.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.46it/s]


Writing augmented_images/caltech256/color_controlnet/005.baseball-glove/005_0138.jpg...


100%|██████████| 22/22 [00:00<00:00, 23.50it/s]

Writing augmented_images/caltech256/color_controlnet/005.baseball-glove/005_0009.jpg...


In [5]:
def initialize_zero123_models():
    """
    Initialize all models and detectors used in the pipeline
    Returns:
        control_net: dict, containing all ControlNet models and detectors
        llava: dict, containing the LLAVA model and processor
        zero123: dict, containing the Zero123 model and Carvekit interface
        color_control: dict, containing the Color Control model and SAM annotator

    """

    # Zero123 models
    # print('Loading Zero123 models...')
    zero123 = {}
    config_path = './model_configs/sd-objaverse-finetune-c_concat-256.yaml'
    config = OmegaConf.load(config_path)

    model_path = "./models/105000.ckpt"
    model = load_model_from_config(config, model_path, zero123_device)
    model = model.to(zero123_device)

    # print('Creating Carvekit interface...')
    carvekit_interface = create_carvekit_interface()

    zero123['model'] = model
    zero123['carvekit_interface'] = carvekit_interface 
    
    return zero123


In [7]:
zero123_model = initialize_zero123_models()
zero123_model

Loading model from ./models/105000.ckpt
Global Step: 105000
LatentDiffusion: Running in eps-prediction mode


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/pytorch_lightning/core/lightning.py:2058: DeprecationWarning: `torch.distributed._sharded_tensor` will be deprecated, use `torch.distributed._shard.sharded_tensor` instead
  from torch.distributed._sharded_tensor import pre_load_state_dict_hook, state_dict_hook


DiffusionWrapper has 859.53 M params.
Keeping EMAs of 688.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels


Downloading: "https://github.com/DagnyT/hardnet/raw/master/pretrained/train_liberty_with_aug/checkpoint_liberty_with_aug.pth" to /home/vaibhav/.cache/torch/hub/checkpoints/checkpoint_liberty_with_aug.pth
100%|██████████| 5.10M/5.10M [00:00<00:00, 35.1MB/s]
100%|███████████████████████████████████████| 890M/890M [00:16<00:00, 58.2MiB/s]
/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/carvekit/ml/wrap/tracer_b7.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via t

{'model': LatentDiffusion(
   (model): DiffusionWrapper(
     (diffusion_model): UNetModel(
       (time_embed): Sequential(
         (0): Linear(in_features=320, out_features=1280, bias=True)
         (1): SiLU()
         (2): Linear(in_features=1280, out_features=1280, bias=True)
       )
       (input_blocks): ModuleList(
         (0): TimestepEmbedSequential(
           (0): Conv2d(8, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
         )
         (1-2): 2 x TimestepEmbedSequential(
           (0): ResBlock(
             (in_layers): Sequential(
               (0): GroupNorm32(32, 320, eps=1e-05, affine=True)
               (1): SiLU()
               (2): Conv2d(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
             )
             (h_upd): Identity()
             (x_upd): Identity()
             (emb_layers): Sequential(
               (0): SiLU()
               (1): Linear(in_features=1280, out_features=320, bias=True)
             )
             (ou

In [ ]:
# Define the angles for augmentation
angles = [
    ("right", 0, 15, 0),
]

from IPython.display import display
import cv2

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

for class_name in classes:

    input_directory = os.path.join("torch/caltech256/256_ObjectCategories", class_name)
    output_directory = os.path.join("augmented_images/caltech256/zero123", class_name)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    count = 0
    for file in os.listdir(input_directory):
        if count >= 2:
            break
        
        img = Image.open(os.path.join(input_directory, file))

        # Apply Zero123 augmentations
        preprocessed_image, augmented_images = generate_angles(
            input_image=img,
            angles=angles,
            model=zero123_model['model'],  # Your loaded model
            carvekit_interface=zero123_model['carvekit_interface'],  # Your Carvekit interface
            device=zero123_device,  # Your device (e.g., 'cuda:0')
            precision='autocast',  # Use 'autocast' for mixed precision if supported
            h=256,
            w=256,
            ddim_steps=50,
            scale=3.0,
            n_samples=1,
            ddim_eta=1.0
        )
        output_path = os.path.join(output_directory, file)

        augmented_images[0].save(output_path)
        print("Writing " + output_path + "...")
        

        count += 1


old input_im: (700, 514)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 46.24it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/001.ak47/001_0072.jpg...
old input_im: (300, 186)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 48.23it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/001.ak47/001_0003.jpg...
old input_im: (237, 235)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


...

Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/004.baseball-bat/004_0049.jpg...
old input_im: (191, 161)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 47.92it/s]


Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/005.baseball-glove/005_0138.jpg...
old input_im: (200, 200)
Generating right view with angles x=0, y=15, z=0
Data shape for DDIM sampling is (1, 4, 32, 32), eta 1.0
Running DDIM Sampling with 49 timesteps


DDIM Sampler: 100%|██████████| 49/49 [00:01<00:00, 48.16it/s]

Sampled tensor shape: torch.Size([1, 4, 32, 32])
Writing augmented_images/caltech256/zero123/005.baseball-glove/005_0009.jpg...


In [2]:
def initialize_models():
    # Initialize ControlNet models
    print('Loading ControlNet models...')
    control_net = {}
    model_names = ['control_v11p_sd15_canny', 'control_v11f1p_sd15_depth', 'control_v11p_sd15_seg']
    models = {}
    for name in model_names:
        model = create_model(f'./model_configs/{name}.yaml').cpu()
        model.load_state_dict(load_state_dict('./models/v1-5-pruned.ckpt', location=control_net_device), strict=False)
        model.load_state_dict(load_state_dict(f'./models/{name}.pth', location=control_net_device), strict=False)
        models[name] = model.to(control_net_device)

    # Initialize Control Netdetectors
    apply_canny = CannyDetector()
    apply_depth = MidasDetector()
    apply_seg = UniformerDetector()
    detectors = {'Canny': apply_canny, 'Depth': apply_depth, 'Segmentation': apply_seg}
    
    control_net['models'] = models
    control_net['detectors'] = detectors
    return control_net

In [3]:
def control_augment(control_net, device, det, input_image, prompt, a_prompt, n_prompt, num_samples, image_resolution, detect_resolution, ddim_steps, guess_mode, strength, scale, seed, eta, low_threshold, high_threshold):
    with torch.no_grad():
        input_image = np.array(input_image)
        input_image = HWC3(input_image)
        input_image = resize_image(input_image, detect_resolution)
        H, W, C = input_image.shape

        if det == 'Canny':
            detected_map = control_net['detectors']['Canny'](input_image, low_threshold, high_threshold)
            model = control_net['models']['control_v11p_sd15_canny']
        elif det == 'Depth':
            # print(control_net['detectors']['Depth'](input_image))
            detected_map = control_net['detectors']['Depth'](input_image)
            model = control_net['models']['control_v11f1p_sd15_depth']
        elif det == 'Segmentation':
            detected_map = control_net['detectors']['Segmentation'](input_image)
            model = control_net['models']['control_v11p_sd15_seg']
        else:
            raise ValueError(f"Unknown detection type: {det}")

        detected_map = HWC3(detected_map)
        detected_map = cv2.resize(detected_map, (W, H), interpolation=cv2.INTER_LINEAR)

        img = resize_image(input_image, image_resolution)
        H, W, C = img.shape

        control = torch.from_numpy(detected_map.copy()).float().to(device) / 255.0
        control = torch.stack([control for _ in range(num_samples)], dim=0)
        control = einops.rearrange(control, 'b h w c -> b c h w').clone()

        if seed == -1:
            seed = random.randint(0, 65535)
        seed_everything(seed)

        model.low_vram_shift(is_diffusing=False)

        cond = {"c_concat": [control], "c_crossattn": [model.get_learned_conditioning([prompt + ', ' + a_prompt] * num_samples)]}
        un_cond = {"c_concat": None if guess_mode else [control], "c_crossattn": [model.get_learned_conditioning([n_prompt] * num_samples)]}
        shape = (4, H // 8, W // 8)

        model.low_vram_shift(is_diffusing=True)

        model.control_scales = [strength * (0.825 ** float(12 - i)) for i in range(13)] if guess_mode else ([strength] * 13)

        ddim_sampler = ControlNetDDIMSampler(model)
        samples, intermediates = ddim_sampler.sample(ddim_steps, num_samples,
                                                     shape, cond, verbose=False, eta=eta,
                                                     unconditional_guidance_scale=scale,
                                                     unconditional_conditioning=un_cond)

        model.low_vram_shift(is_diffusing=False)

        x_samples = model.decode_first_stage(samples)
        x_samples = (einops.rearrange(x_samples, 'b c h w -> b h w c') * 127.5 + 127.5).cpu().numpy().clip(0, 255).astype(np.uint8)

        results = [x_samples[i] for i in range(num_samples)]
    return detected_map, results[0]


In [4]:
control_net = initialize_models()

Loading ControlNet models...
ControlLDM: Running in eps-prediction mode
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.


/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/pytorch_lightning/core/lightning.py:2058: DeprecationWarning: `torch.distributed._sharded_tensor` will be deprecated, use `torch.distributed._shard.sharded_tensor` instead
  from torch.distributed._sharded_tensor import pre_load_state_dict_hook, state_dict_hook


Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setti

/home/vaibhav/miniconda3/envs/diffaug/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at openai/clip-vit-large-patch14 were not used when initializing CLIPTextModel: ['vision_model.encoder.layers.9.layer_norm2.weight', 'vision_model.encoder.layers.1.mlp.fc1.weight', 'vision_model.encoder.layers.13.self_attn.v_proj.bias', 'vision_model.encoder.layers.14.layer_norm2.bias', 'vision_model.encoder.layers.13.self_attn.k_proj.weight', 'vision_model.encoder.layers.2.layer_norm1.bias', 'vision_model.encoder.layers.2.mlp.fc2.bias', 'vision_model.encoder.layers.7.layer_norm1.weight', 'vision_model.encoder.layers.13.self_attn.out_proj.bias', 'vision_model.encoder.layers.16.layer_norm2.weight', 'vision_model.encoder.layers.18.layer_norm1.wei

Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is 768 and using 8 heads.
Setting up Me

...

100%|██████████| 470M/470M [00:06<00:00, 82.0MB/s] 
/home/vaibhav/diffusion_augmentation/controlnet/annotator/midas/midas/base_model.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimenta

Downloading: "https://huggingface.co/lllyasviel/Annotators/resolve/main/upernet_global_small.pth" to /home/vaibhav/diffusion_augmentation/controlnet/annotator/ckpts/upernet_global_small.pth



100%|██████████| 197M/197M [00:02<00:00, 81.1MB/s] 


Use Checkpoint: False
Checkpoint Number: [0, 0, 0, 0]
Use global window for all blocks in stage3
load checkpoint from local path: /home/vaibhav/diffusion_augmentation/controlnet/annotator/ckpts/upernet_global_small.pth


/home/vaibhav/diffusion_augmentation/controlnet/annotator/uniformer/mmcv/runner/checkpoint.py:262: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(file

In [ ]:
from IPython.display import display
import cv2

classes = ["001.ak47", "002.american-flag", "003.backpack", "004.baseball-bat", "005.baseball-glove"]

for class_name in classes:

    caption = class_name.split(".")[-1]
    input_directory = os.path.join("torch/caltech256/256_ObjectCategories", class_name)

    count = 0
    for file in os.listdir(input_directory):
        if count >= 2:
            break
        
        img = Image.open(os.path.join(input_directory, file))

        augmented_images = {}
        preprocessed_images = {}
        for det_type in ['Canny', 'Depth', 'Segmentation']:
            print(f"Processing {det_type}...")
            preprocessed, augmented = control_augment(
                control_net=control_net,
                device=control_net_device,
                det=det_type,
                input_image=img,
                prompt=caption,
                a_prompt="clear image, photorealistic",
                n_prompt="multiple, mushed, low quality, cropped, worst quality",
                num_samples=1,
                image_resolution=512,
                detect_resolution=512,
                ddim_steps=20,
                guess_mode=False,
                strength=1.0,
                scale=7.5,
                seed=-1,
                eta=0.0,
                low_threshold=100,
                high_threshold=200,
                
            )
            augmented_images[det_type] = Image.fromarray(augmented)
            preprocessed_images[det_type] = Image.fromarray(preprocessed)

            output_path = os.path.join("augmented_images/caltech256", det_type, class_name, file)
            if not os.path.exists(os.path.dirname(output_path)):
                os.makedirs(os.path.dirname(output_path))

            augmented_images[det_type].save(output_path)
            print("Writing " + output_path + "...")

        count += 1

Global seed set to 22767


Processing Canny...
Data shape for DDIM sampling is (1, 4, 64, 64), eta 0.0
Running DDIM Sampling with 20 timesteps


DDIM Sampler: 100%|██████████| 20/20 [00:04<00:00,  4.36it/s]
Global seed set to 18742


Writing augmented_images/caltech256/Canny/005.baseball-glove/005_0052.jpg...
Processing Depth...
Data shape for DDIM sampling is (1, 4, 64, 64), eta 0.0
Running DDIM Sampling with 20 timesteps


DDIM Sampler: 100%|██████████| 20/20 [00:04<00:00,  4.32it/s]
/home/vaibhav/diffusion_augmentation/controlnet/annotator/uniformer/mmseg/models/segmentors/base.py:271: UserWarning: show==False and out_file is not specified, only result image will be returned
  warnings.warn('show==False and out_file is not specified, only '
Global seed set to 18693


...

Writing augmented_images/caltech256/Canny/005.baseball-glove/005_0049.jpg...
Processing Depth...
Data shape for DDIM sampling is (1, 4, 64, 88), eta 0.0
Running DDIM Sampling with 20 timesteps


DDIM Sampler: 100%|██████████| 20/20 [00:06<00:00,  2.87it/s]
Global seed set to 58125


Writing augmented_images/caltech256/Depth/005.baseball-glove/005_0049.jpg...
Processing Segmentation...
Data shape for DDIM sampling is (1, 4, 64, 88), eta 0.0
Running DDIM Sampling with 20 timesteps


DDIM Sampler: 100%|██████████| 20/20 [00:06<00:00,  2.87it/s]


Writing augmented_images/caltech256/Segmentation/005.baseball-glove/005_0049.jpg...
